# Invoking MCP-based Strands Agents in Amazon Bedrock AgentCore Runtime from AWS Lambda with CloudWatch Observability

## Overview

This tutorial demonstrates how to invoke a Strands agent that uses Model Context Protocol (MCP) servers hosted in Amazon Bedrock AgentCore Runtime from an AWS Lambda function, with full CloudWatch observability enabled.

### Tutorial Details

| Information         | Details                                                                          |
|:-------------------|:----------------------------------------------------------------------------------|
| Tutorial type      | Conversational                                                                   |
| Agent type         | Single                                                                           |
| Agentic Framework  | Strands Agents                                                                   |
| LLM model          | Anthropic Claude Sonnet 3.7                                                      |
| Tutorial components| Lambda invocation, AgentCore Runtime, MCP servers, CloudWatch Observability     |
| Example complexity | Advanced                                                                         |
| SDK used           | Amazon BedrockAgentCore Python SDK, boto3, AWS Lambda                           |

### Tutorial Architecture

```
API/User → AWS Lambda → AgentCore Runtime → Strands Agent → MCP Servers (AWS Docs + CDK)
                ↓                                    ↓
          CloudWatch                          CloudWatch
          (X-Ray Traces)                      (Gen AI Observability)
```

### Key Features

* Integrating multiple MCP servers (AWS Documentation + AWS CDK) with Strands Agents
* Hosting agents on Amazon Bedrock AgentCore Runtime
* Invoking hosted agents from AWS Lambda functions
* Configuring CloudWatch Gen AI Observability for agent monitoring
* Enabling AWS X-Ray tracing for Lambda functions
* Viewing traces, spans, and metrics in CloudWatch console

### What You'll Learn

1. How to deploy an MCP-enabled agent to AgentCore Runtime
2. How to create a Lambda function that invokes the runtime agent
3. How to configure X-Ray sampling for Lambda observability
4. How to enable CloudWatch Gen AI Observability for your agents
5. How to view and analyze traces showing agent execution flow

## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials configured with appropriate permissions
* Amazon Bedrock AgentCore SDK
* Strands Agents with OTEL support
* MCP libraries
* Permissions to create Lambda functions and IAM roles
* CloudWatch Transaction Search enabled (we'll do this in the tutorial)

Let's install the required packages:

In [ ]:
!pip install --upgrade "strands-agents[otel]" strands-agents-tools boto3 bedrock-agentcore bedrock-agentcore-starter-toolkit uv

## Step 1: Enable CloudWatch Transaction Search (One-Time Setup)

Before we can use CloudWatch Gen AI Observability, we need to enable Transaction Search in CloudWatch. This is a one-time setup per AWS account.

**Important**: After enabling Transaction Search, it can take up to 10 minutes for spans to become available for search and analysis.

In [ ]:
import boto3
import json
from boto3.session import Session

# Initialize session and get account details
boto_session = Session()
region = boto_session.region_name
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']

print(f"AWS Account ID: {account_id}")
print(f"Region: {region}")

In [ ]:
# Step 1.1: Create CloudWatch Logs resource policy for X-Ray
logs_client = boto3.client('logs', region_name=region)

policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "TransactionSearchXRayAccess",
            "Effect": "Allow",
            "Principal": {
                "Service": "xray.amazonaws.com"
            },
            "Action": "logs:PutLogEvents",
            "Resource": [
                f"arn:aws:logs:{region}:{account_id}:log-group:aws/spans:*",
                f"arn:aws:logs:{region}:{account_id}:log-group:/aws/application-signals/data:*"
            ],
            "Condition": {
                "ArnLike": {
                    "aws:SourceArn": f"arn:aws:xray:{region}:{account_id}:*"
                },
                "StringEquals": {
                    "aws:SourceAccount": account_id
                }
            }
        }
    ]
}

try:
    logs_client.put_resource_policy(
        policyName='TransactionSearchXRayPolicy',
        policyDocument=json.dumps(policy_document)
    )
    print("✅ CloudWatch Logs resource policy created successfully")
except logs_client.exceptions.LimitExceededException:
    print("⚠️ Resource policy already exists or limit exceeded")
except Exception as e:
    print(f"⚠️ Policy creation note: {e}")

In [ ]:
# Step 1.2: Configure X-Ray to send trace segments to CloudWatch Logs
xray_client = boto3.client('xray', region_name=region)

try:
    xray_client.update_trace_segment_destination(
        Destination='CloudWatchLogs'
    )
    print("✅ X-Ray trace segment destination configured to CloudWatch Logs")
except Exception as e:
    print(f"⚠️ Note: {e}")

In [ ]:
# Step 1.3: Configure span indexing sampling percentage (1% is free tier)
try:
    xray_client.update_indexing_rule(
        Name='Default',
        Rule={
            'Probabilistic': {
                'DesiredSamplingPercentage': 1.0  # 1% sampling for free tier
            }
        }
    )
    print("✅ X-Ray indexing rule configured (1% sampling)")
except Exception as e:
    print(f"⚠️ Note: {e}")

print("\n⏰ Please wait 10 minutes for Transaction Search to be fully enabled before viewing traces.")

## Step 2: Create MCP Agent with Multiple Servers

We'll create an agent that uses two MCP servers:
1. AWS Documentation MCP Server - for accessing AWS documentation
2. AWS CDK MCP Server - for CDK best practices and guidance

This agent will be instrumented with OpenTelemetry for full observability.

In [ ]:
%%writefile mcp_agent_multi_server.py
from strands import Agent
from strands.models import BedrockModel
from mcp import StdioServerParameters, stdio_client
from strands.tools.mcp import MCPClient
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Initialize the BedrockAgentCoreApp
app = BedrockAgentCoreApp()

# Connect to AWS Documentation MCP server
def create_aws_docs_client():
    return MCPClient(
        lambda: stdio_client(
            StdioServerParameters(
                command="uvx", 
                args=["awslabs.aws-documentation-mcp-server@latest"]
            )
        )
    )

# Connect to AWS CDK MCP server
def create_cdk_client():
    return MCPClient(
        lambda: stdio_client(
            StdioServerParameters(
                command="uvx", 
                args=["awslabs.cdk-mcp-server@latest"]
            )
        )
    )

# Function to create agent with tools from both MCP servers
def create_agent():
    model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    model = BedrockModel(model_id=model_id)
    
    aws_docs_client = create_aws_docs_client()
    cdk_client = create_cdk_client()
    
    with aws_docs_client, cdk_client:
        # Get tools from both MCP servers
        tools = aws_docs_client.list_tools_sync() + cdk_client.list_tools_sync()
        
        # Create agent with these tools
        agent = Agent(
            model=model,
            tools=tools,
            system_prompt="""You are a helpful AWS assistant with access to AWS Documentation 
            and CDK best practices. Provide concise and accurate information about AWS services 
            and infrastructure as code patterns. When asked about pricing or CDK, use your tools 
            to search for the most current information."""
        )
    
    return agent, aws_docs_client, cdk_client

@app.entrypoint
def invoke_agent(payload):
    """Process the input payload and return the agent's response"""
    agent, aws_docs_client, cdk_client = create_agent()
    
    with aws_docs_client, cdk_client:
        user_input = payload.get("prompt")
        print(f"Processing request: {user_input}")
        response = agent(user_input)
        return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

In [ ]:
%%writefile requirements.txt
strands-agents[otel]
strands-agents-tools
uv
boto3
bedrock-agentcore
aws-opentelemetry-distro>=0.10.0

## Step 3: Deploy Agent to AgentCore Runtime

We'll deploy the MCP agent to AgentCore Runtime. The runtime automatically instruments the agent with OpenTelemetry, enabling CloudWatch Gen AI Observability.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Configure the runtime
agentcore_runtime = Runtime()
agent_name = "mcp_agent_lambda_observability"

# Configure the agent runtime
config_response = agentcore_runtime.configure(
    entrypoint="mcp_agent_multi_server.py",
    auto_create_ecr=True,
    auto_create_execution_role=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)

print(f"✅ Agent configured: {agent_name}")
config_response

In [ ]:
# Launch the agent to AgentCore Runtime
print("🚀 Deploying agent to AgentCore Runtime (this may take several minutes)...")
launch_result = agentcore_runtime.launch()

print(f"\n✅ Agent deployed successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

# Save agent details for Lambda function
agent_arn = launch_result.agent_arn
agent_id = launch_result.agent_id

In [ ]:
# Wait for agent to be ready
import time

print("⏳ Waiting for agent endpoint to be ready...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_statuses = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_statuses:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Status: {status}")

if status == 'READY':
    print("\n✅ Agent is ready to accept invocations!")
else:
    print(f"\n❌ Agent deployment failed with status: {status}")

# Get the endpoint ARN
endpoint_arn = status_response.endpoint['agentRuntimeEndpointArn']
print(f"Endpoint ARN: {endpoint_arn}")

## Step 4: Test Direct Invocation (Before Lambda)

Let's first test the agent directly from the notebook to ensure it's working correctly.

In [ ]:
# Test direct invocation
test_payload = {"prompt": "What is Amazon Bedrock's pricing model?"}

print(f"Testing agent with prompt: {test_payload['prompt']}\n")
invoke_response = agentcore_runtime.invoke(test_payload)

# Display response
from IPython.display import Markdown, display
response_text = invoke_response['response'][0]
display(Markdown(response_text))

## Step 5: Create Lambda Execution Role with X-Ray Tracing

The Lambda function needs permissions to:
1. Invoke the AgentCore Runtime agent
2. Write logs to CloudWatch
3. Send traces to X-Ray

In [ ]:
import json

iam_client = boto3.client('iam')

# Define Lambda execution role name
lambda_role_name = f"AgentCoreLambdaExecutionRole-{agent_name}"

# Trust policy for Lambda
lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "lambda.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

# Create or get the role
try:
    role_response = iam_client.create_role(
        RoleName=lambda_role_name,
        AssumeRolePolicyDocument=json.dumps(lambda_trust_policy),
        Description='Execution role for Lambda to invoke AgentCore Runtime with X-Ray tracing'
    )
    lambda_role_arn = role_response['Role']['Arn']
    print(f"✅ Created new Lambda execution role: {lambda_role_arn}")
    time.sleep(10)  # Wait for role to propagate
except iam_client.exceptions.EntityAlreadyExistsException:
    role_response = iam_client.get_role(RoleName=lambda_role_name)
    lambda_role_arn = role_response['Role']['Arn']
    print(f"✅ Using existing Lambda execution role: {lambda_role_arn}")

# Attach AWS managed policies
managed_policies = [
    'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',  # CloudWatch Logs
    'arn:aws:iam::aws:policy/AWSXRayDaemonWriteAccess',  # X-Ray tracing
]

for policy_arn in managed_policies:
    try:
        iam_client.attach_role_policy(
            RoleName=lambda_role_name,
            PolicyArn=policy_arn
        )
        print(f"✅ Attached policy: {policy_arn.split('/')[-1]}")
    except Exception as e:
        print(f"⚠️ Policy already attached: {policy_arn.split('/')[-1]}")

In [ ]:
# Create custom policy for AgentCore Runtime invocation
agentcore_policy_name = f"AgentCoreRuntimeInvokePolicy-{agent_name}"

# Get current endpoint ARN
status_response = agentcore_runtime.status()
if 'agentRuntimeEndpointArn' in status_response.endpoint:
    current_endpoint_arn = status_response.endpoint['agentRuntimeEndpointArn']
else:
    current_endpoint_arn = status_response.endpoint.get('endpointArn', endpoint_arn)

print(f"Creating policy with resources:")
print(f"  Agent ARN: {agent_arn}")
print(f"  Endpoint ARN: {current_endpoint_arn}")

agentcore_policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AgentCoreRuntimeAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:InvokeAgentRuntime",
                "bedrock-agentcore:GetAgentRuntime",
                "bedrock-agentcore:GetAgentRuntimeEndpoint",
                "bedrock-agentcore:ListAgentRuntimes"
            ],
            "Resource": [
                agent_arn,
                f"{agent_arn}/*",
                current_endpoint_arn,
                f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/*"
            ]
        },
        {
            "Sid": "BedrockModelAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}

try:
    policy_response = iam_client.create_policy(
        PolicyName=agentcore_policy_name,
        PolicyDocument=json.dumps(agentcore_policy_document),
        Description='Policy to allow Lambda to invoke AgentCore Runtime'
    )
    agentcore_policy_arn = policy_response['Policy']['Arn']
    print(f"✅ Created AgentCore invocation policy: {agentcore_policy_arn}")
except iam_client.exceptions.EntityAlreadyExistsException:
    agentcore_policy_arn = f"arn:aws:iam::{account_id}:policy/{agentcore_policy_name}"
    print(f"✅ Using existing AgentCore invocation policy: {agentcore_policy_arn}")

# Attach custom policy to role
try:
    iam_client.attach_role_policy(
        RoleName=lambda_role_name,
        PolicyArn=agentcore_policy_arn
    )
    print(f"✅ Attached AgentCore invocation policy to Lambda role")
except Exception as e:
    print(f"⚠️ Policy already attached")

print(f"\n✅ Lambda role fully configured with X-Ray and AgentCore permissions")
print(f"\nGranted permissions:")
print(f"  • InvokeAgentRuntime, GetAgentRuntime, GetAgentRuntimeEndpoint")
print(f"  • Bedrock model invocation")
print(f"  • Wildcard access to all runtimes in account")

## Step 6: Create Lambda Function to Invoke AgentCore Runtime

This Lambda function will:
1. Accept a prompt in the event payload
2. Invoke the AgentCore Runtime agent
3. Return the agent's response
4. Automatically generate X-Ray traces for observability

In [ ]:
%%writefile lambda_agentcore_invoker.py
import json
import boto3
import os
import traceback
from botocore.exceptions import ClientError

def lambda_handler(event, context):
    """
    Lambda function to invoke AgentCore Runtime agent.
    
    Expected event format:
    {
        "prompt": "Your question here",
        "sessionId": "optional-session-id"  # Optional
    }
    """
    
    # Initialize boto3 client inside handler for better Lambda performance
    bedrock_agentcore_client = boto3.client('bedrock-agentcore')
    
    try:
        # Get environment variables
        agent_arn = os.environ.get('AGENT_ARN')
        endpoint_arn = os.environ.get('ENDPOINT_ARN')
        
        print(f"Lambda function started")
        print(f"Agent ARN: {agent_arn}")
        print(f"Endpoint ARN: {endpoint_arn}")
        
        if not agent_arn or not endpoint_arn:
            return {
                'statusCode': 500,
                'body': json.dumps({
                    'error': 'Configuration Error',
                    'message': 'Missing AGENT_ARN or ENDPOINT_ARN environment variables'
                })
            }
        
        # Parse input
        if isinstance(event, str):
            event = json.loads(event)
        
        prompt = event.get('prompt', '')
        session_id = event.get('sessionId', context.aws_request_id)
        
        if not prompt:
            return {
                'statusCode': 400,
                'body': json.dumps({
                    'error': 'Bad Request',
                    'message': 'Missing prompt in request'
                })
            }
        
        print(f"Processing prompt: {prompt}")
        print(f"Session ID: {session_id}")
        
        # Prepare payload for AgentCore
        payload = json.dumps({"prompt": prompt})
        
        # Invoke AgentCore Runtime
        print(f"Invoking AgentCore Runtime...")
        response = bedrock_agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=endpoint_arn,
            runtimeSessionId=session_id,
            payload=payload
        )
        
        print(f"Response received from AgentCore")
        print(f"Response keys: {list(response.keys())}")
        
        # Parse response - handle StreamingBody
        agent_response = None
        
        if 'response' in response:
            response_body = response['response']
            print(f"Response body type: {type(response_body).__name__}")
            
            # Check if it's a StreamingBody (has read method)
            if hasattr(response_body, 'read'):
                # It's a StreamingBody, read it
                raw_data = response_body.read()
                print(f"Raw data type after read: {type(raw_data).__name__}")
                
                if isinstance(raw_data, bytes):
                    agent_response = raw_data.decode('utf-8')
                else:
                    agent_response = str(raw_data)
                    
            elif isinstance(response_body, list) and len(response_body) > 0:
                # It's a list
                if isinstance(response_body[0], bytes):
                    agent_response = response_body[0].decode('utf-8')
                else:
                    agent_response = str(response_body[0])
                    
            elif isinstance(response_body, bytes):
                agent_response = response_body.decode('utf-8')
                
            elif isinstance(response_body, str):
                agent_response = response_body
                
            else:
                agent_response = str(response_body)
        
        if not agent_response:
            agent_response = "No response from agent"
            print("Warning: No response extracted from AgentCore")
        
        print(f"Agent response received (length: {len(agent_response)} chars)")
        print(f"Agent response preview: {agent_response[:100]}...")
        
        return {
            'statusCode': 200,
            'body': json.dumps({
                'response': agent_response,
                'sessionId': session_id,
                'agentArn': agent_arn
            }),
            'headers': {
                'Content-Type': 'application/json'
            }
        }
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        error_message = e.response['Error']['Message']
        print(f"AWS ClientError: {error_code}")
        print(f"Error message: {error_message}")
        traceback.print_exc()
        
        return {
            'statusCode': 500,
            'body': json.dumps({
                'error': error_code,
                'message': error_message
            })
        }
    
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error: {str(e)}")
        traceback.print_exc()
        
        return {
            'statusCode': 400,
            'body': json.dumps({
                'error': 'Invalid JSON',
                'message': str(e)
            })
        }
    
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        print(f"Error type: {type(e).__name__}")
        traceback.print_exc()
        
        return {
            'statusCode': 500,
            'body': json.dumps({
                'error': 'InternalError',
                'message': str(e),
                'type': type(e).__name__
            })
        }

In [ ]:
# Create deployment package
import zipfile
import os

# Create a zip file with the Lambda function code
zip_filename = 'lambda_agentcore_invoker.zip'

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('lambda_agentcore_invoker.py')

print(f"✅ Created Lambda deployment package: {zip_filename}")

# Read the zip file
with open(zip_filename, 'rb') as f:
    lambda_zip_content = f.read()

print(f"Package size: {len(lambda_zip_content)} bytes")

In [ ]:
# Create Lambda function with CORRECT endpoint ARN
lambda_client = boto3.client('lambda', region_name=region)
lambda_function_name = f"agentcore-mcp-invoker-{agent_name}"

# Get the CORRECT endpoint ARN (agentRuntimeArn, NOT agentRuntimeEndpointArn)
status_response = agentcore_runtime.status()
correct_endpoint_arn = status_response.endpoint['agentRuntimeArn']

print(f"📋 Lambda Configuration:")
print(f"  Function Name: {lambda_function_name}")
print(f"  Agent ARN:     {agent_arn}")
print(f"  Endpoint ARN:  {correct_endpoint_arn}")
print(f"  AWS Region:    {region}")
print()

lambda_config = {
    'FunctionName': lambda_function_name,
    'Runtime': 'python3.12',
    'Role': lambda_role_arn,
    'Handler': 'lambda_agentcore_invoker.lambda_handler',
    'Code': {'ZipFile': lambda_zip_content},
    'Description': 'Lambda function to invoke AgentCore Runtime with MCP servers',
    'Timeout': 300,  # 5 minutes
    'MemorySize': 512,
    'Environment': {
        'Variables': {
            'AGENT_ARN': agent_arn,
            'ENDPOINT_ARN': correct_endpoint_arn,  # 수정: agentRuntimeArn 사용
        }
    },
    'TracingConfig': {
        'Mode': 'Active'  # Enable X-Ray tracing
    }
}

try:
    lambda_response = lambda_client.create_function(**lambda_config)
    lambda_function_arn = lambda_response['FunctionArn']
    print(f"✅ Created Lambda function: {lambda_function_name}")
    print(f"Function ARN: {lambda_function_arn}")
except lambda_client.exceptions.ResourceConflictException:
    # Function already exists, update it
    print(f"⚠️  Function exists, updating...")
    
    lambda_client.update_function_code(
        FunctionName=lambda_function_name,
        ZipFile=lambda_zip_content
    )
    print(f"  ✅ Code updated")
    
    lambda_client.update_function_configuration(
        FunctionName=lambda_function_name,
        Environment=lambda_config['Environment'],
        TracingConfig=lambda_config['TracingConfig'],
        Timeout=lambda_config['Timeout'],
        MemorySize=lambda_config['MemorySize']
    )
    print(f"  ✅ Configuration updated")
    
    lambda_function_arn = lambda_client.get_function(FunctionName=lambda_function_name)['Configuration']['FunctionArn']
    print(f"✅ Updated existing Lambda function: {lambda_function_name}")
    print(f"Function ARN: {lambda_function_arn}")

# Wait for function to be ready
print("\n⏳ Waiting for Lambda function to be active...")
waiter = lambda_client.get_waiter('function_active_v2')
waiter.wait(FunctionName=lambda_function_name)
print("✅ Lambda function is active and ready!")

# Verify the configuration
print("\n🔍 Verifying environment variables:")
verify_config = lambda_client.get_function_configuration(FunctionName=lambda_function_name)
verify_env = verify_config['Environment']['Variables']
print(f"  AGENT_ARN:    {verify_env['AGENT_ARN']}")
print(f"  ENDPOINT_ARN: {verify_env['ENDPOINT_ARN']}")

## Step 7: Configure X-Ray Sampling for Lambda

X-Ray sampling controls what percentage of requests are traced. We'll configure sampling to capture traces for analysis in CloudWatch.

In [ ]:
# Verify X-Ray tracing is enabled
lambda_config = lambda_client.get_function_configuration(
    FunctionName=lambda_function_name
)

tracing_mode = lambda_config['TracingConfig']['Mode']
print(f"Lambda X-Ray Tracing Mode: {tracing_mode}")

if tracing_mode == 'Active':
    print("✅ X-Ray tracing is ENABLED for Lambda function")
    print("   - Lambda will automatically send traces to X-Ray")
    print("   - Traces will be available in CloudWatch Transaction Search")
    print("   - You can view traces in CloudWatch Gen AI Observability dashboard")
else:
    print("⚠️ X-Ray tracing is NOT enabled")
    print("   Run the following AWS CLI command to enable it:")
    print(f"   aws lambda update-function-configuration --function-name {lambda_function_name} --tracing-config Mode=Active --region {region}")

## Step 8: Invoke Lambda Function and Generate Traces

Now let's invoke the Lambda function to generate traces that we can view in CloudWatch.

In [ ]:
# Test payload for Lambda invocation
test_payloads = [
    # {
    #     "prompt": "What is Amazon Bedrock's pricing model? Be concise."
    # },
    # {
    #     "prompt": "What are CDK best practices for Lambda functions?"
    # },
    {
        "prompt": "How do I use AWS Step Functions with Lambda?"
    }
]

print("🚀 Invoking Lambda function with test payloads...\n")

for i, payload in enumerate(test_payloads, 1):
    print(f"\n{'='*80}")
    print(f"Test {i}: {payload['prompt']}")
    print('='*80)
    
    try:
        # Invoke Lambda function
        response = lambda_client.invoke(
            FunctionName=lambda_function_name,
            InvocationType='RequestResponse',
            Payload=json.dumps(payload)
        )
        
        # Parse response
        response_payload = json.loads(response['Payload'].read())
        
        # Check if Lambda execution itself failed (FunctionError)
        if 'FunctionError' in response:
            print(f"\n❌ Lambda Function Error!")
            print(f"Error Type: {response_payload.get('errorType', 'Unknown')}")
            print(f"Error Message: {response_payload.get('errorMessage', 'Unknown')}")
            
            # Print stack trace if available
            if 'stackTrace' in response_payload:
                print(f"\nStack Trace:")
                for line in response_payload['stackTrace'][:5]:  # First 5 lines
                    print(f"  {line.strip()}")
            continue
        
        # Check if response has expected structure
        if 'statusCode' not in response_payload:
            print(f"\n⚠️ Unexpected response structure:")
            print(json.dumps(response_payload, indent=2))
            continue
            
        # Process successful response
        if response_payload['statusCode'] == 200:
            body = json.loads(response_payload['body'])
            print(f"\n✅ Success!")
            print(f"Session ID: {body.get('sessionId', 'N/A')}")
            
            response_text = body.get('response', '')
            if len(response_text) > 500:
                print(f"\nAgent Response:\n{response_text[:500]}...")  # Show first 500 chars
            else:
                print(f"\nAgent Response:\n{response_text}")
        else:
            # Handle error responses from Lambda
            print(f"\n❌ Error Response (Status: {response_payload['statusCode']})")
            try:
                body = json.loads(response_payload['body'])
                print(f"Error: {body.get('error', 'Unknown')}")
                print(f"Message: {body.get('message', 'Unknown')}")
            except:
                print(f"Raw Response: {response_payload.get('body', 'No body')}")
    
    except json.JSONDecodeError as e:
        print(f"\n❌ JSON Decode Error: {e}")
        print(f"Raw payload: {response['Payload'].read()}")
    
    except KeyError as e:
        print(f"\n❌ Missing key in response: {e}")
        print(f"Available keys: {response_payload.keys() if 'response_payload' in locals() else 'N/A'}")
        print(f"Full response: {json.dumps(response_payload, indent=2)}")
    
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
    
    # Small delay between requests
    time.sleep(2)

print("\n" + "="*80)
print("✅ All test invocations completed!")
print("\n⏰ Traces are being processed and will be available in CloudWatch within 1-2 minutes.")

## Step 9: View Observability Data in CloudWatch

Now that we've generated traces, let's explore how to view them in CloudWatch.

### Where to Find Your Observability Data

#### 1. CloudWatch Gen AI Observability Dashboard
This is your primary dashboard for viewing agent performance:

- **URL**: [CloudWatch Gen AI Observability Console](https://console.aws.amazon.com/cloudwatch/home#genai-observability)
- **What you'll see**:
  - Agents View: List of all your agents with runtime metrics
  - Sessions View: All sessions across agents
  - Traces View: Detailed trace information with span timelines

#### 2. Lambda X-Ray Traces
View Lambda-specific traces:

- **URL**: [X-Ray Service Map](https://console.aws.amazon.com/xray/home#/service-map)
- **What you'll see**:
  - Service map showing Lambda → AgentCore connections
  - Request traces with detailed timing
  - Error rates and latency percentiles

#### 3. CloudWatch Logs
Raw logs from Lambda and AgentCore:

- **Lambda Logs**: `/aws/lambda/{lambda_function_name}`
- **AgentCore Logs**: `/aws/bedrock-agentcore/runtimes/{agent_id}`

Let's generate the direct URLs:

In [ ]:
from IPython.display import display, HTML

# Generate CloudWatch console URLs
base_url = f"https://{region}.console.aws.amazon.com/cloudwatch"

urls = {
    "Gen AI Observability Dashboard": f"{base_url}/home?region={region}#genai-observability",
    "Lambda Function Logs": f"{base_url}/home?region={region}#logsV2:log-groups/log-group/$252Faws$252Flambda$252F{lambda_function_name}",
    "AgentCore Runtime Logs": f"{base_url}/home?region={region}#logsV2:log-groups/log-group/$252Faws$252Fbedrock-agentcore$252Fruntimes$252F{agent_id}",
    "X-Ray Service Map": f"https://{region}.console.aws.amazon.com/xray/home?region={region}#/service-map",
    "X-Ray Traces": f"https://{region}.console.aws.amazon.com/xray/home?region={region}#/traces",
    "Transaction Search": f"{base_url}/home?region={region}#application-signals/transaction-search"
}

html_content = "<h3>CloudWatch Observability Links</h3><ul>"
for name, url in urls.items():
    html_content += f'<li><strong>{name}</strong>: <a href="{url}" target="_blank">{url}</a></li>'
html_content += "</ul>"

display(HTML(html_content))

print("\n📊 Key Metrics to Look For:")
print("\n1. In Gen AI Observability Dashboard:")
print("   - Session duration and count")
print("   - Token usage (input/output)")
print("   - MCP tool invocation traces")
print("   - Error rates")
print("\n2. In X-Ray Traces:")
print("   - Lambda → AgentCore latency")
print("   - AgentCore → Bedrock model calls")
print("   - End-to-end request duration")
print("   - Service map showing all connections")
print("\n3. In CloudWatch Logs:")
print("   - Lambda execution logs")
print("   - Agent processing steps")
print("   - MCP server communication logs")
print("   - OpenTelemetry span data")

## Step 10: Query Traces Programmatically

Let's use the AWS SDK to query and display trace data.

In [ ]:
from datetime import datetime, timedelta

xray_client = boto3.client('xray', region_name=region)

# Query traces from the last 30 minutes
end_time = datetime.utcnow()
start_time = end_time - timedelta(minutes=30)

print(f"Querying X-Ray traces from {start_time} to {end_time}...\n")

try:
    # Get trace summaries
    trace_summaries = xray_client.get_trace_summaries(
        StartTime=start_time,
        EndTime=end_time,
        FilterExpression=f'service("{lambda_function_name}")'
    )
    
    traces = trace_summaries.get('TraceSummaries', [])
    
    if traces:
        print(f"✅ Found {len(traces)} traces for Lambda function\n")
        
        # Display summary of traces
        for i, trace in enumerate(traces[:5], 1):  # Show first 5 traces
            print(f"Trace {i}:")
            print(f"  ID: {trace.get('Id')}")
            print(f"  Duration: {trace.get('Duration', 0):.2f} seconds")
            print(f"  Response Time: {trace.get('ResponseTime', 0):.2f} seconds")
            print(f"  HTTP Status: {trace.get('Http', {}).get('HttpStatus', 'N/A')}")
            
            # Check for errors
            if trace.get('HasError'):
                print(f"  ⚠️ Has Error: {trace.get('ErrorRootCauses')}")
            else:
                print(f"  ✅ No errors")
            print()
        
        # Get detailed trace data for the first trace
        if traces:
            first_trace_id = traces[0]['Id']
            print(f"\nFetching detailed data for trace: {first_trace_id}...\n")
            
            trace_detail = xray_client.batch_get_traces(
                TraceIds=[first_trace_id]
            )
            
            if trace_detail.get('Traces'):
                segments = trace_detail['Traces'][0].get('Segments', [])
                print(f"Found {len(segments)} segments in this trace:\n")
                
                for segment in segments:
                    doc = json.loads(segment['Document'])
                    print(f"  Segment: {doc.get('name')}")
                    print(f"    Origin: {doc.get('origin', 'N/A')}")
                    print(f"    Duration: {doc.get('end_time', 0) - doc.get('start_time', 0):.3f}s")
                    
                    # Check for subsegments (like AgentCore calls)
                    subsegments = doc.get('subsegments', [])
                    if subsegments:
                        print(f"    Subsegments: {len(subsegments)}")
                        for subseg in subsegments[:3]:  # Show first 3 subsegments
                            print(f"      - {subseg.get('name')}: {subseg.get('end_time', 0) - subseg.get('start_time', 0):.3f}s")
                    print()
    else:
        print("⚠️ No traces found yet. This could be because:")
        print("   1. Traces are still being processed (wait 1-2 minutes)")
        print("   2. Lambda function hasn't been invoked yet")
        print("   3. X-Ray sampling is filtering out traces")
        print("\n💡 Try running the Lambda invocations again and wait a couple minutes.")

except Exception as e:
    print(f"Error querying traces: {e}")
    print("\nThis might be because:")
    print("   - Transaction Search is still being set up (wait 10 minutes after enabling)")
    print("   - Insufficient permissions to query X-Ray")
    print("   - No traces have been generated yet")

## Step 11: Query CloudWatch Logs for Agent Observability

Let's query the CloudWatch Logs to see the agent's execution details.

In [ ]:
logs_client = boto3.client('logs', region_name=region)

# Lambda logs
lambda_log_group = f"/aws/lambda/{lambda_function_name}"

print(f"Querying CloudWatch Logs for Lambda function...\n")

try:
    # Get recent log streams
    log_streams = logs_client.describe_log_streams(
        logGroupName=lambda_log_group,
        orderBy='LastEventTime',
        descending=True,
        limit=5
    )
    
    if log_streams['logStreams']:
        print(f"✅ Found {len(log_streams['logStreams'])} recent log streams\n")
        
        # Get logs from the most recent stream
        latest_stream = log_streams['logStreams'][0]
        stream_name = latest_stream['logStreamName']
        
        print(f"Getting logs from stream: {stream_name}\n")
        
        log_events = logs_client.get_log_events(
            logGroupName=lambda_log_group,
            logStreamName=stream_name,
            limit=20
        )
        
        print("Recent Lambda Execution Logs:")
        print("="*80)
        
        for event in log_events['events']:
            timestamp = datetime.fromtimestamp(event['timestamp'] / 1000)
            message = event['message'].strip()
            
            # Highlight important log messages
            if 'START' in message:
                print(f"\n🚀 {timestamp}: {message}")
            elif 'END' in message:
                print(f"✅ {timestamp}: {message}")
            elif 'ERROR' in message or 'Error' in message:
                print(f"❌ {timestamp}: {message}")
            elif 'Invoking agent' in message:
                print(f"📞 {timestamp}: {message}")
            elif 'response received' in message.lower():
                print(f"📥 {timestamp}: {message}")
            else:
                print(f"   {timestamp}: {message}")
        
        print("\n" + "="*80)
    else:
        print("⚠️ No log streams found yet. Invoke the Lambda function first.")

except logs_client.exceptions.ResourceNotFoundException:
    print(f"⚠️ Log group not found: {lambda_log_group}")
    print("   This will be created automatically when Lambda executes.")
except Exception as e:
    print(f"Error querying logs: {e}")

In [ ]:
# Query AgentCore Runtime logs
agentcore_log_group = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"

print(f"\nQuerying CloudWatch Logs for AgentCore Runtime...\n")

try:
    # Get recent log streams for AgentCore
    log_streams = logs_client.describe_log_streams(
        logGroupName=agentcore_log_group,
        orderBy='LastEventTime',
        descending=True,
        limit=5
    )
    
    if log_streams['logStreams']:
        print(f"✅ Found {len(log_streams['logStreams'])} AgentCore log streams\n")
        
        # Filter for runtime-logs stream (contains OTEL data)
        runtime_streams = [s for s in log_streams['logStreams'] if 'runtime-logs' in s['logStreamName']]
        
        if runtime_streams:
            stream_name = runtime_streams[0]['logStreamName']
            print(f"Getting OpenTelemetry logs from: {stream_name}\n")
            
            log_events = logs_client.get_log_events(
                logGroupName=agentcore_log_group,
                logStreamName=stream_name,
                limit=10
            )
            
            print("AgentCore Runtime Observability Logs:")
            print("="*80)
            
            for event in log_events['events']:
                timestamp = datetime.fromtimestamp(event['timestamp'] / 1000)
                message = event['message'].strip()
                
                # Try to parse as JSON for structured logs
                try:
                    log_data = json.loads(message)
                    
                    # Extract relevant observability fields
                    if 'body' in log_data:
                        body = log_data['body']
                        trace_id = log_data.get('trace_id', 'N/A')
                        span_id = log_data.get('span_id', 'N/A')
                        
                        print(f"\n📊 {timestamp}")
                        print(f"   Trace ID: {trace_id}")
                        print(f"   Span ID: {span_id}")
                        print(f"   Message: {body[:200]}...")  # First 200 chars
                    else:
                        print(f"\n   {timestamp}: {json.dumps(log_data, indent=2)[:200]}...")
                        
                except json.JSONDecodeError:
                    # Not JSON, just print the message
                    print(f"\n   {timestamp}: {message[:200]}...")
            
            print("\n" + "="*80)
        else:
            print("⚠️ No runtime-logs stream found. Check log stream names.")
    else:
        print("⚠️ No AgentCore log streams found yet.")
        
except logs_client.exceptions.ResourceNotFoundException:
    print(f"⚠️ Log group not found: {agentcore_log_group}")
    print("   AgentCore logs should be created automatically.")
except Exception as e:
    print(f"Error querying AgentCore logs: {e}")

## Step 12: View Key Observability Metrics

Let's query CloudWatch Metrics to see performance data for our Lambda function and agent.

In [ ]:
cloudwatch = boto3.client('cloudwatch', region_name=region)

# Define time range for metrics
end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=1)

print("📊 Querying CloudWatch Metrics...\n")

# Lambda metrics
lambda_metrics = [
    ('Invocations', 'Count'),
    ('Duration', 'Milliseconds'),
    ('Errors', 'Count'),
    ('Throttles', 'Count'),
]

print("Lambda Function Metrics:")
print("="*80)

for metric_name, unit in lambda_metrics:
    try:
        response = cloudwatch.get_metric_statistics(
            Namespace='AWS/Lambda',
            MetricName=metric_name,
            Dimensions=[
                {'Name': 'FunctionName', 'Value': lambda_function_name}
            ],
            StartTime=start_time,
            EndTime=end_time,
            Period=300,  # 5 minutes
            Statistics=['Sum', 'Average', 'Maximum']
        )
        
        datapoints = response.get('Datapoints', [])
        
        if datapoints:
            latest = sorted(datapoints, key=lambda x: x['Timestamp'])[-1]
            print(f"\n{metric_name}:")
            print(f"  Sum: {latest.get('Sum', 0):.2f} {unit}")
            print(f"  Average: {latest.get('Average', 0):.2f} {unit}")
            print(f"  Maximum: {latest.get('Maximum', 0):.2f} {unit}")
        else:
            print(f"\n{metric_name}: No data available yet")
            
    except Exception as e:
        print(f"\n{metric_name}: Error querying - {e}")

print("\n" + "="*80)

## Summary and Best Practices

### What We Accomplished

✅ **Agent Deployment**: Created and deployed an MCP agent with multiple servers (AWS Docs + CDK) to AgentCore Runtime

✅ **Lambda Integration**: Built a Lambda function that invokes the hosted agent as a thin invocation layer

✅ **X-Ray Tracing**: Enabled AWS X-Ray tracing on Lambda for request-level observability

✅ **CloudWatch Gen AI Observability**: Configured Transaction Search and enabled full agent observability

✅ **Trace Generation**: Generated sample traces showing the complete flow: Lambda → AgentCore → MCP Servers

✅ **Observability Dashboard**: Set up access to view traces, metrics, and logs in CloudWatch console

### Key Observability Features

1. **X-Ray Traces**: See the complete request path from Lambda through AgentCore to MCP servers
2. **CloudWatch Logs**: Detailed execution logs with timestamps and session IDs
3. **OpenTelemetry Data**: Structured span and trace data following Gen AI semantic conventions
4. **Performance Metrics**: Duration, token usage, error rates, and throughput
5. **Session Tracking**: Correlate multiple requests within the same conversation

### Observability Best Practices

1. **Use Session IDs**: Pass consistent session IDs to correlate related requests
2. **Monitor Key Metrics**: Track duration, token usage, and error rates
3. **Set Up Alarms**: Create CloudWatch alarms for error rates and latency thresholds
4. **Sampling Strategy**: Adjust X-Ray sampling based on traffic volume (1% is free tier)
5. **Log Retention**: Configure appropriate retention periods for cost optimization
6. **Trace Analysis**: Regularly review traces to identify bottlenecks and optimize performance

### Sample Configuration for X-Ray

The Lambda function was configured with:
```python
TracingConfig: {'Mode': 'Active'}  # Enables X-Ray tracing
```

You can also configure this via AWS CLI:
```bash
aws lambda update-function-configuration \
  --function-name {lambda_function_name} \
  --tracing-config Mode=Active \
  --region {region}
```

### Next Steps

1. **Explore the Dashboard**: Visit the CloudWatch Gen AI Observability dashboard using the links above
2. **Analyze Traces**: Click on individual traces to see the execution timeline and MCP tool calls
3. **Create Alarms**: Set up CloudWatch alarms for error rates or high latency
4. **Optimize Performance**: Use trace data to identify and optimize slow operations
5. **Scale Monitoring**: As traffic grows, adjust sampling rates and configure log aggregation

### Cleanup (Optional)

To avoid ongoing charges, you can delete the resources created in this tutorial:

In [ ]:
# Uncomment and run to delete resources

# # Delete Lambda function
# lambda_client.delete_function(FunctionName=lambda_function_name)
# print(f"✅ Deleted Lambda function: {lambda_function_name}")

# # Delete Lambda execution role
# # First detach policies
# for policy_arn in managed_policies + [agentcore_policy_arn]:
#     try:
#         iam_client.detach_role_policy(RoleName=lambda_role_name, PolicyArn=policy_arn)
#     except:
#         pass
        
# iam_client.delete_role(RoleName=lambda_role_name)
# print(f"✅ Deleted Lambda role: {lambda_role_name}")

# # Delete custom policy
# iam_client.delete_policy(PolicyArn=agentcore_policy_arn)
# print(f"✅ Deleted custom policy")

# # Delete AgentCore Runtime agent
# agentcore_runtime.delete()
# print(f"✅ Deleted AgentCore Runtime agent")

print("To delete resources, uncomment the code above and run this cell.")

## Congratulations! 🎉

You have successfully:
- Deployed an MCP-enabled agent to Amazon Bedrock AgentCore Runtime
- Created a Lambda function to invoke the agent with X-Ray tracing enabled
- Configured CloudWatch Gen AI Observability for comprehensive monitoring
- Generated and viewed traces showing the complete agent execution flow

You can now use these patterns to build production-ready AI agents with full observability!